In [76]:
# === Imports ===
import os
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import to_rgb
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import TwoSlopeNorm
from matplotlib import rcParams
import copy
import matplotlib.lines as mlines
from plotting_utils import parse_filename, make_label, MODEL_PALETTE  # reuse your existing utility functions
from matplotlib.ticker import MaxNLocator
from contextlib import contextmanager

# === Config ===
FOLDER_PATH = '.'  # Path to folder containing CSVs

PRESENTATION_MODE = False  # Toggle for presentation mode (white text, transparent bg)
SAVE_FORMATS = ['png']  # Output formats

# Save original rcParams for later restore
ORIGINAL_RCPARAMS = copy.deepcopy(rcParams)

def get_marker(row):
    if row["style"] == 10:
        if row["context"] == 1:
            return "o" if row["ft"] == 1 else "d"
        else:
            return "s"
    else:
        return "^"

def apply_presentation_style():
    """Applies consistent rcParams for plotting."""
    base_fontsize = 16
    color = "white" if PRESENTATION_MODE else "black"
    
    rcParams.update({
        "axes.facecolor": "none" if PRESENTATION_MODE else "white",
        "figure.facecolor": "none" if PRESENTATION_MODE else "white",
        "text.color": color,
        "xtick.color": color,
        "ytick.color": color,
        "axes.labelcolor": color,
        "axes.edgecolor": color,
        "legend.edgecolor": color,
        "font.size": base_fontsize,
        "axes.titlesize": base_fontsize + 4,
        "axes.labelsize": base_fontsize,
        "xtick.labelsize": base_fontsize,
        "ytick.labelsize": base_fontsize,
        "legend.fontsize": base_fontsize,
        "legend.title_fontsize": base_fontsize + 2,
    })

@contextmanager
def with_plot_style():
    """Context manager to apply and restore plot style."""
    original = copy.deepcopy(rcParams)
    apply_presentation_style()
    yield
    rcParams.update(original)

In [125]:
RESULTS_FOLDER = "Figures/features_differences"
EXCLUDED_FEATURES = [
    'spelling_grammar_errors', 'has_link', 'has_emoji', 'has_mention',
    # 'has_question_mark', 'has_exclamation_mark', 
]


def compute_feature_differences(df, label_col='labels'):
    grouped = df.groupby(label_col).mean()
    if 0 in grouped.index and 1 in grouped.index:
        return (grouped.loc[0] - grouped.loc[1]).to_dict()
    else:
        return {}


def process_response_type(folder_path, response_type, label_col="labels"):
    differences_list = []
    model_names = set()
    results_folder = os.path.join(folder_path, RESULTS_FOLDER, response_type)
    os.makedirs(results_folder, exist_ok=True)

    for root, _, files in os.walk(folder_path):
        for filename in files:
            if response_type not in filename or not filename.endswith('validation_data_features.csv'):
                continue

            features_path = os.path.join(root, filename)
            labels_path = features_path.replace('features.csv', 'labelled.csv')

            if not os.path.exists(labels_path):
                print(f"Skipping: no corresponding label file for {features_path}")
                continue

            df_features = pd.read_csv(features_path)
            df_labels = pd.read_csv(labels_path)

            if len(df_features) != len(df_labels):
                print(f"Length mismatch: {filename}")
                continue

            if label_col not in df_labels.columns:
                print(f"'{label_col}' column missing in {labels_path}")
                continue

            df = pd.concat([df_features, df_labels[label_col]], axis=1)
            model, ft, context, style, oppu = parse_filename(filename)
            model_names.add(model)

            numeric_features = df.select_dtypes(include='number').drop(columns=[label_col])
            df_numeric = pd.concat([numeric_features, df[label_col]], axis=1)

            diffs = compute_feature_differences(df_numeric, label_col)
            if not diffs:
                continue

            diffs.update({
                'model': model,
                'oppu': oppu,
                'ft': ft,
                'context': context,
                'style': style,
            })
            differences_list.append(diffs)

    if not differences_list:
        print(f"No differences computed for {response_type}.")
        return None, None

    diff_df = pd.DataFrame(differences_list)
    sort_cols = ["model", "oppu", "ft", "context", "style"]
    diff_df = diff_df.set_index(sort_cols)
    return diff_df, results_folder


def plot_feature_differences_bar(diff_df, results_folder, response_type):
    
    diff_df['abs_total_diff'] = diff_df.abs().sum(axis=1)
    diff_df_sorted = diff_df.sort_values(by='abs_total_diff')
    diff_df_features = diff_df_sorted.drop(columns='abs_total_diff')
    diff_df_features = diff_df_features.drop(columns=EXCLUDED_FEATURES, errors='ignore')

    for feature in diff_df_features.columns:
        feature_values = []
        for idx, row in diff_df_features.iterrows():
            model, oppu, ft, context, style = idx
            config_label = make_label(model, ft, context, style, oppu)
            val = row[feature]
            feature_values.append((config_label, val, model))

        feature_values_sorted = sorted(feature_values, key=lambda x: abs(x[1]))
        labels = [x[0] for x in feature_values_sorted]
        bars = [x[1] for x in feature_values_sorted]
        colors = [MODEL_PALETTE.get(x[2], 'gray') for x in feature_values_sorted]

        with with_plot_style():
            plt.figure(figsize=(max(10, len(labels) * 0.4), 6))
            plt.bar(labels, bars, color=colors)
            plt.title(f"{feature} — Mean Difference ({response_type})", fontsize=16)
            plt.ylabel("Difference", fontsize=14)
            plt.xticks(rotation=45, ha='right', fontsize=10)
            plt.tight_layout()
            path = os.path.join(results_folder, f"feature_diff_{feature}.png")
            plt.savefig(path, dpi=600, transparent=PRESENTATION_MODE)
            plt.close()

def plot_feature_differences_heatmap(diff_df, results_folder):

    # Prepare data
    heatmap_df = diff_df.copy().reset_index()
    heatmap_df = heatmap_df.sort_values(by=['model', 'style', 'context', 'ft'])

    heatmap_df['config'] = heatmap_df.apply(
        lambda row: make_label(row['model'], row['ft'], row['context'], row['style'], row['oppu']),
        axis=1
    )

    heatmap_df['config'] = pd.Categorical(
        heatmap_df['config'],
        categories=heatmap_df.sort_values('model')['config'],
        ordered=True
    )

    columns_to_drop = ['ft', 'context', 'style', 'oppu', 'has_link', 'has_mention', 'has_emoji', 'abs_total_diff']
    heatmap_data = heatmap_df.drop(columns=columns_to_drop).set_index('config')
    models_for_colors = heatmap_df.set_index('config')['model']

    # Save raw differences for later (to color annotations)
    raw_values = heatmap_data.drop(columns='model')

    # Z-score per feature
    scaler = StandardScaler()
    heatmap_data_normalized = pd.DataFrame(
        scaler.fit_transform(raw_values),
        index=heatmap_data.index,
        columns=[col.replace('_', ' ') for col in raw_values.columns]
    )
    # Use a custom diverging colormap with stronger saturation
    cmap = mpl.cm.get_cmap("coolwarm")  # Alternative: "seismic" or "PiYG"
    norm = TwoSlopeNorm(vmin=heatmap_data_normalized.min().min(),
                        vcenter=0,
                        vmax=heatmap_data_normalized.max().max())

    # Plot
    with with_plot_style():
        plt.figure(figsize=(14, max(6, 0.3 * len(heatmap_data_normalized))))
        ax = sns.heatmap(
            heatmap_data_normalized,
            cmap='vlag',
            center=0,
            cbar_kws={'label': 'Z-scored (per feature)'},
            fmt='',
            annot_kws={'size': 8},
            annot=False  # We'll manually add annotations below
        )

        # Colorbar settings
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=14)
        cbar.set_label("Z-scored (per feature)", fontsize=16)

        # Set label colors based on model palette
        for label in ax.get_yticklabels():
            config_label = label.get_text()
            model_name = models_for_colors.loc[config_label]
            label.set_color(MODEL_PALETTE.get(model_name, 'white'))

        # Add sign-colored annotations
        for i in range(heatmap_data_normalized.shape[0]):
            for j in range(heatmap_data_normalized.shape[1]):
                val = raw_values.iloc[i, j]
                text_color = "red" if val > 0 else "blue" if val < 0 else "black"
                ax.text(j + 0.5, i + 0.5, f"{val:.2f}",
                        ha='center', va='center', color=text_color, fontsize=8)

        # Labels & layout
        ax.set_xlabel('')
        ax.set_ylabel('')
            
        plt.title("Normalized Feature Differences", fontsize=20)
        ax.set_xticklabels(ax.get_xticklabels(), fontsize=13)
        apply_presentation_style()
        _style_axis(ax, models_for_colors)
        plt.tight_layout()

        path = os.path.join(results_folder, "feature_differences_heatmap.png")
        plt.savefig(path, dpi=600, transparent=PRESENTATION_MODE)
        plt.close()

def plot_difference_between_responses(diff_optimal, diff_random, FOLDER_PATH):
    """Generate raw and normalized heatmaps of feature differences between optimal and random responses."""

    # Align indices
    shared_idx = diff_optimal.index.intersection(diff_random.index)
    if shared_idx.empty:
        print("No overlapping configurations between optimal and random.")
        return

    delta_df = diff_optimal.loc[shared_idx] - diff_random.loc[shared_idx]
    delta_df = delta_df.reset_index()

    # Sort and generate config labels
    delta_df = delta_df.sort_values(by=['model', 'style', 'context', 'ft'])
    delta_df['config'] = delta_df.apply(
        lambda row: make_label(row['model'], row['ft'], row['context'], row['style'], row['oppu']),
        axis=1
    )
    delta_df = delta_df.drop(columns=EXCLUDED_FEATURES, errors='ignore')
    delta_df = delta_df.drop(columns='abs_total_diff', errors='ignore')
    

    config_to_model = delta_df.set_index('config')['model']
    feature_cols = [
        col for col in delta_df.columns
        if col not in {'model', 'ft', 'context', 'style', 'oppu', 'config'}
        and pd.api.types.is_numeric_dtype(delta_df[col])
    ]
    delta_values = delta_df.set_index('config')[feature_cols]
    delta_values = delta_values.loc[config_to_model.sort_values().index]
    models_for_colors = config_to_model.loc[delta_values.index]

    with with_plot_style():
        fig, ax = plt.subplots(figsize=(14, max(6, 0.3 * len(delta_values))))
        sns.heatmap(
            delta_values,
            cmap='vlag',
            center=0,
            annot=False,
            fmt='.2f',
            annot_kws={'size': 8},
            cbar_kws={'label': 'Feature Difference (Optimal − Random)'},
            ax=ax
        )

        # Add sign-colored annotations
        for i in range(delta_values.shape[0]):
            for j in range(delta_values.shape[1]):
                val = delta_values.iloc[i, j]
                text_color = "red" if val > 0 else "blue" if val < 0 else "black"
                ax.text(j + 0.5, i + 0.5, f"{val:.2f}",
                        ha='center', va='center', color=text_color, fontsize=8)
                

        _style_axis(ax, models_for_colors)
        ax.set_title("Feature Differences (Optimal − Random)")

        apply_presentation_style()
        fig.tight_layout()
        fig.savefig(os.path.join(RESULTS_FOLDER, "delta_feature_differences_heatmap_raw.png"), dpi=600, transparent=PRESENTATION_MODE)
        plt.close(fig)

def _style_axis(ax, models_for_colors):
    """Standardize x/y axis formatting for heatmaps."""
    # Clean x-tick labels
    ax.set_xticklabels(
        [label.get_text().replace('_', ' ') for label in ax.get_xticklabels()],
        fontsize=13,
        rotation=45,
        ha='right'
    )

    # Color y-tick labels by model
    ytick_labels = ax.get_yticklabels()
    for label in ytick_labels:
        config_label = label.get_text()
        model_name = models_for_colors.get(config_label)
        label.set_color(MODEL_PALETTE.get(model_name, 'white'))
    ax.set_yticklabels(ytick_labels, fontsize=13)

    ax.set_xlabel('')
    ax.set_ylabel('')

    # Style colorbar
    if ax.collections:
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=14)
        cbar.set_label(cbar.ax.get_ylabel(), fontsize=16)

In [126]:
diffs = {}

for response in ['random', 'optimal']:
    diff_df, results_folder = process_response_type(FOLDER_PATH, response_type=response)
    if diff_df is not None:
        plot_feature_differences_bar(diff_df, results_folder, response_type=response)
        plot_feature_differences_heatmap(diff_df, results_folder)
        diffs[response] = diff_df

if 'optimal' in diffs and 'random' in diffs:
    plot_difference_between_responses(diffs['optimal'], diffs['random'], RESULTS_FOLDER)

/var/folders/nx/vkg436q12hd36l9v_4tzqh180000gn/T/ipykernel_31451/564401536.py:138: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = mpl.cm.get_cmap("coolwarm")  # Alternative: "seismic" or "PiYG"
/var/folders/nx/vkg436q12hd36l9v_4tzqh180000gn/T/ipykernel_31451/564401536.py:138: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = mpl.cm.get_cmap("coolwarm")  # Alternative: "seismic" or "PiYG"


# FEATURES

In [ ]:
def collect_auc_data(label_source):
    """Collect AUC data for both random and optimal responses."""
    auc_records = []
    suffix = "_from_labels_feature_importance_stats.csv" if label_source == "labels" else "_from_bert_feature_importance_stats.csv"
    
    for response_type in ['random_response', 'optimal_response']:
        for root, _, files in os.walk(FOLDER_PATH):
            for filename in files:
                if response_type not in filename:
                    continue
                if filename.endswith(suffix):
                    full_path = os.path.join(root, filename)
                    df = pd.read_csv(full_path)

                    model = df['model'].iloc[0]
                    ft = df['ft'].iloc[0]
                    context = df['context'].iloc[0]
                    style = df['style'].iloc[0]
                    oppu = df['oppu'].iloc[0]
                    auc = df['auc'].iloc[0]

                    auc_records.append({
                        'model': model,
                        'ft': ft,
                        'context': context,
                        'style': style,
                        'oppu': oppu,
                        'response_type': response_type,
                        'auc': auc
                    })

    return pd.DataFrame(auc_records)


def plot_bar_chart(data, values_col, title, ylabel, filename, label_col='label', color_col='color', save_folder=None):
    """Generic bar chart plotting function with consistent style and saving."""
  
    with with_plot_style():
        fig, ax = plt.subplots(figsize=(14, 6))
        bars = ax.bar(data[label_col], data[values_col], color=data[color_col])

        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_title(title, fontsize=16)
        ax.set_xticks(range(len(data[label_col])))
        ax.set_xticklabels(data[label_col], rotation=45)
        ax.grid(True, axis='y')

        # Legend by model
        unique_models = data['model'].unique()
        handles = [
            plt.Line2D([0], [0], marker='o', color='w', label=model,
                    markerfacecolor=MODEL_PALETTE.get(model, 'gray'), markersize=10)
            for model in unique_models
        ]
        apply_presentation_style()

        plt.tight_layout()
        if save_folder:
            os.makedirs(save_folder, exist_ok=True)
            save_path = os.path.join(save_folder, filename)
            plt.savefig(save_path, dpi=600, bbox_inches='tight', transparent=PRESENTATION_MODE)
        plt.close()


def plot_auc_summary(auc_df_all, label_source, save_folder):
    """Generate and save AUC bar plots and scatter plot comparing optimal vs. random responses."""
    if auc_df_all.empty:
        print("No AUC data found.")
        return

    sort_cols = ['model', 'ft', 'context', 'style', 'oppu']
    auc_pivot = auc_df_all.pivot_table(index=sort_cols, columns='response_type', values='auc').reset_index()
    auc_pivot = auc_pivot.rename(columns={'random_response': 'auc_random', 'optimal_response': 'auc_optimal'})
    auc_pivot = auc_pivot.sort_values(by=['model', 'style', 'context', 'ft'])
    auc_pivot['label'] = [
        make_label(row['model'], row['ft'], row['context'], row['style'], row['oppu'])
        for _, row in auc_pivot.iterrows()
    ]
    auc_pivot['color'] = auc_pivot['model'].map(MODEL_PALETTE)
    labels = auc_pivot['label'].tolist()
    plot_suffix = 'from_labels' if label_source == 'labels' else 'from_bert'

    # Plot AUC for optimal and random
    plot_bar_chart(auc_pivot, 'auc_optimal', f'AUC Score for Optimal Responses',
             'AUC', f'auc_optimal_response_{plot_suffix}.png', save_folder=save_folder)
    plot_bar_chart(auc_pivot, 'auc_random', f'AUC Score for Random Responses',
             'AUC', f'auc_random_response_{plot_suffix}.png', save_folder=save_folder)

    # Plot difference
    auc_pivot['auc_diff'] = auc_pivot['auc_optimal'] - auc_pivot['auc_random']
    plot_bar_chart(auc_pivot, 'auc_diff', 'Difference in AUC Score per Model Configuration',
             'AUC Difference (Optimal - Random)', f'auc_difference_{plot_suffix}.png', save_folder=save_folder)

    # --- SCATTER PLOT: AUC DIFF vs. FRACTION MATCH ---
    # Step 1: Collect fraction data from all response_comparisons.csv
    fraction_results = []
    for root, _, files in os.walk(FOLDER_PATH):
        for file in files:
            if file.endswith("response_comparisons.csv"):
                full_path = os.path.join(root, file)
                try:
                    df = pd.read_csv(full_path)
                except Exception as e:
                    print(f"Skipping file {full_path}: {e}")
                    continue
                if 'previous_response' in df.columns and 'selected_response' in df.columns:
                    match_count = (df['previous_response'] == df['selected_response']).sum()
                    total_count = len(df)
                    model, ft, context, style, oppu = parse_filename(file)
                    fraction = match_count / total_count if total_count else 0
                    fraction_results.append({
                        'model': model, 'ft': ft, 'context': context,
                        'style': style, 'oppu': oppu, 'fraction': fraction
                    })

    fraction_df = pd.DataFrame(fraction_results)
    merge_keys = ['model', 'ft', 'context', 'style', 'oppu']
    combined_df = pd.merge(auc_pivot, fraction_df, on=merge_keys, how='inner')
    combined_df['marker'] = combined_df.apply(get_marker, axis=1)
    combined_df['color'] = combined_df['model'].map(MODEL_PALETTE)

    # Step 2: Scatter Plot
    with with_plot_style():
        fig, ax = plt.subplots(figsize=(10, 7))
        for marker in combined_df['marker'].unique():
            subset = combined_df[combined_df['marker'] == marker]
            for model in subset['model'].unique():
                sub = subset[subset['model'] == model]
                ax.scatter(
                    sub['auc_diff'], sub['fraction'],
                    color=MODEL_PALETTE.get(model, 'gray'),
                    marker=marker, s=500, alpha=0.8,
                    edgecolor='k'
                )


        ax.set_xlabel('AUC Difference (Optimal - Random)', color="white" if PRESENTATION_MODE else "black")
        ax.set_ylabel('Fraction of Matches (Optimal = Random)', color="white" if PRESENTATION_MODE else "black")
        ax.set_title('Scatter Plot: AUC Difference vs Fraction of Matches', color="white" if PRESENTATION_MODE else "black")
        ax.grid(True)

        # Legend 1: Model
        model_handles = [
            plt.Line2D([0], [0], marker='o', color='w', label=model,
                    markerfacecolor=color, markersize=10, markeredgecolor='k', linestyle='None')
            for model, color in MODEL_PALETTE.items()
            if model in combined_df['model'].unique()
        ]
        legend1 = ax.legend(handles=model_handles, title='Model', loc='upper left',
                            frameon=True, framealpha=0.1, facecolor='black', edgecolor='white')
        for text in legend1.get_texts():
            text.set_color("white" if PRESENTATION_MODE else "black")
        legend1.get_title().set_color(color="white" if PRESENTATION_MODE else "black")

        # Legend 2: Marker types
        marker_labels = {
            'o': 'SE + CR + FT', #'Style 10, Context 3, FT 1',
            'd': 'SE + CR', #'Style 10, Context 3, FT 0',
            's': 'SE', #'Style 10, Context 0, FT 0',
            '^': 'baseline' #'Style 0, Context 0, FT 0',
        }
        marker_handles = [
            mlines.Line2D([], [], color="white" if PRESENTATION_MODE else "black", 
                        marker=marker, linestyle='None',
                        markersize=10, label=label)
            for marker, label in marker_labels.items()
        ]

        legend2 = ax.legend(handles=marker_handles, title='Marker Type', loc='upper right',
                            frameon=True, framealpha=0.1, facecolor='black', edgecolor='white')
        for text in legend2.get_texts():
            text.set_color(color="white" if PRESENTATION_MODE else "black")
        legend2.get_title().set_color(color="white" if PRESENTATION_MODE else "black")
        ax.add_artist(legend1)
        apply_presentation_style()  # e.g., set background, font sizes, etc.
        plt.tight_layout()
        plt.savefig(os.path.join(save_folder, f'scatter_auc_diff_vs_fraction_{plot_suffix}.png'),
                    dpi=600, bbox_inches='tight', transparent=PRESENTATION_MODE)
        plt.close()

def plot_feature_importance_heatmap(folder_path, results_folder, response_type, suffix, plot_suffix):
    importance_dfs = []

    # Step 1: Collect matching files
    for root, _, files in os.walk(folder_path):
        for filename in files:
            if filename.endswith(suffix) and response_type in filename:
                df = pd.read_csv(os.path.join(root, filename))
                importance_dfs.append(df)

    if not importance_dfs:
        print(f"No feature importance data found for {response_type}.")
        return

    # Step 2: Concatenate and prepare
    importance_df = pd.concat(importance_dfs, ignore_index=True)
    importance_df['config'] = importance_df.apply(
        lambda row: make_label(row['model'], row['ft'], row['context'], row['style'], row['oppu']), axis=1
    )

    # Step 3: Select feature columns
    exclude_cols = [
        'model', 'ft', 'context', 'style', 'oppu', 'label',
        'auc', 'accuracy', 'f1', 'precision', 'recall', 'config',
        #'has_question_mark', 'has_exclamation_mark'
    ]
    feature_cols = [
        col for col in importance_df.columns
        if col not in exclude_cols and pd.api.types.is_numeric_dtype(importance_df[col])
    ]

    # Step 4: Build heatmap matrix
    heatmap_df = importance_df[['config', 'model'] + feature_cols].copy()
    heatmap_data = heatmap_df.set_index('config')[feature_cols]
    heatmap_data = heatmap_data.div(heatmap_data.sum(axis=1), axis=0)
    heatmap_data = heatmap_data[heatmap_data.mean().sort_values(ascending=False).index]

    # Step 5: Sort configs and recolor
    config_to_model = heatmap_df.set_index('config')['model']
    sorted_configs = (
        heatmap_df[['config', 'model']]
        .sort_values(['model', 'config'])['config']
        .tolist()
    )
    heatmap_data = heatmap_data.loc[sorted_configs]
    models_for_colors = config_to_model.loc[heatmap_data.index]

    # Step 6: Plot
    with with_plot_style():
        print(max(6, 0.3 * len(heatmap_data)))
        fig, ax = plt.subplots(figsize=(14, max(6, 0.3 * len(heatmap_data))))
        sns.heatmap(
            heatmap_data,
            annot=True,
            cmap='viridis',
            fmt=".2f",
            ax=ax,
            cbar_kws={'label': 'Relative Importance'},
            linewidths=0.5,
            linecolor='gray',
            annot_kws={'size': 8},
        )

        # Step 7: Style axis
        ax.set_xticklabels([label.get_text().replace("_", " ") for label in ax.get_xticklabels()], fontsize=13, rotation=45, ha='right')
        
        new_yticklabels = []
        for label in ax.get_yticklabels():
            config = label.get_text()
            clean_label = config.replace("_", " ")
            model = models_for_colors.get(config, None)
            label.set_text(clean_label)
            if model:
                label.set_color(MODEL_PALETTE.get(model, "black"))
            new_yticklabels.append(label)
        ax.set_yticklabels(new_yticklabels, fontsize=13)

        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_title(f"Normalized Feature Importance")

        # Step 8: Style colorbar
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=13)
        cbar.set_label("Relative Importance", fontsize=16)
        apply_presentation_style()  # e.g., set background, font sizes, etc.
        # Step 9: Save
        plt.tight_layout()
        filename = f'feature_importance_heatmap_{response_type}_{plot_suffix}.png'
        plt.savefig(os.path.join(RESULTS_FOLDER, filename), dpi=600, bbox_inches='tight', transparent=PRESENTATION_MODE)
    

In [93]:
RESULTS_FOLDER = "Figures/features"
os.makedirs(RESULTS_FOLDER, exist_ok=True)
label_source = 'labels'
suffix = "_from_labels_feature_importance_stats.csv" if label_source == "labels" else "_from_bert_feature_importance_stats.csv"
plot_suffix = "from_labels" if label_source == "labels" else "from_bert"

# Step 1: Load AUC data
auc_df_all = collect_auc_data(label_source)

# Step 2: Plot
plot_auc_summary(auc_df_all, label_source, save_folder=RESULTS_FOLDER)

# Step 3: Plot Heatmaps
for response_type in ['random_response', 'optimal_response']:
    plot_feature_importance_heatmap(FOLDER_PATH, RESULTS_FOLDER, response_type, suffix, plot_suffix)

8.4
8.4


# ACCURACY

In [123]:
RESULTS_FOLDER =  "Figures/accuracy"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

# === Data Loading ===
def load_confusion_data(folder_path):
    results = []

    for root, _, files in os.walk(folder_path):
        for file in files:
            if not file.endswith("confusion_matrix.csv"):
                continue
            filepath = os.path.join(root, file)
        
            if "optimal_response" in filepath:
                response_type = "optimal"
            elif "random_response" in filepath:
                response_type = "random"
            else:
                continue

            try:
                df = pd.read_csv(filepath)
                cm = df.values
                tn, fp = cm[0, 0], cm[0, 1]
                fn, tp = cm[1, 0], cm[1, 1]

                total = tn + fp + fn + tp
                correct = tn + tp
                total_0 = tn + fp
                correct_0 = tn

                model, ft, context, style, oppu = parse_filename(file)

                results.append({
                    "model": model,
                    "style": style,
                    "context": context,
                    "ft": ft,
                    "oppu": oppu,
                    "accuracy": correct / total,
                    "class_0_accuracy": correct_0 / total_0 if total_0 > 0 else None,
                    "label": make_label(model, ft, context, style, oppu),
                    "short_label": make_label(model, ft, context, style, oppu, False),
                    "response_type": response_type
                })
            except Exception as e:
                print(f"Failed to process {file}: {e}")

    return pd.DataFrame(results)

# === Plotting ===
def save_figure(fig, name):
    path = os.path.join(RESULTS_FOLDER, f"{name}.png")
    fig.savefig(path, transparent=PRESENTATION_MODE)

def plot_all_accuracy_tiled(df):
    apply_presentation_style()

    with with_plot_style():
        fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)

        positions = [
            ("random", "accuracy", "Overall Accuracy"),
            ("random", "class_0_accuracy", "Class-0 (AI-generated text) Accuracy"),
            ("optimal", "accuracy", "Overall Accuracy"),
            ("optimal", "class_0_accuracy", "Class-0 (AI-generated text) Accuracy"),
        ]

        all_handles = []
        all_labels = []

        for i, (response_type, metric, title) in enumerate(positions):
            row, col = divmod(i, 2)
            ax = axes[row][col]
            subset = df[df["response_type"] == response_type].copy()

            # Sort and prepare
            model_order = {model: k for k, model in enumerate(sorted(subset["model"].unique()))}
            subset["model_order"] = subset["model"].map(model_order)
            sort_cols = ["model_order", "oppu", "ft", "context", "style"]
            subset = subset.sort_values(by=sort_cols)
            subset["label"] = pd.Categorical(subset["label"], categories=subset["label"], ordered=True)

            sns.barplot(
                ax=ax,
                x="label", y=metric,
                hue="model", data=subset,
                dodge=False,
                palette=MODEL_PALETTE
            )

            label_to_short = dict(zip(subset["label"], subset["short_label"]))
            ax.set_xticklabels([label_to_short.get(lbl.get_text(), lbl.get_text()) for lbl in ax.get_xticklabels()], rotation=90)
            ax.set_ylim(0.0, 1.0)
            ax.set_title(f"{title} ({response_type})", pad=10)
            ax.set_xlabel("")
            ax.set_ylabel("")

            # Capture handles and labels only once (avoid duplicates)
            if not all_handles:
                handles, labels = ax.get_legend_handles_labels()
                all_handles, all_labels = handles, labels

            ax.get_legend().remove()

        # Deduplicate legend
        from collections import OrderedDict
        legend_dict = OrderedDict(zip(all_labels, all_handles))

        legend = fig.legend(
            legend_dict.values(), legend_dict.keys(),
            loc='upper right',
            bbox_to_anchor=(0.45, 0.9),
            title="Model",
            fontsize="small",
            title_fontsize="small",
            ncol=2,
        )
        legend.get_frame().set_alpha(0.7)

        fig.tight_layout()
        save_figure(fig, "accuracy_grid_all")
        plt.close(fig)

def plot_accuracy_by_response_type(df):
    apply_presentation_style()

    response_types = ['random', 'optimal']
    metric_titles = {
        'accuracy': "Overall Accuracy",
        'class_0_accuracy': "Class-0 (AI-generated text) Accuracy",
    }

    for response_type in response_types:
        with with_plot_style():
            fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True, sharey=True)

            all_handles = []
            all_labels = []

            for i, (metric, title) in enumerate(metric_titles.items()):
                ax = axes[i]
                subset = df[df["response_type"] == response_type].copy()

                # Sort models for consistent color ordering
                model_order = {model: k for k, model in enumerate(sorted(subset["model"].unique()))}
                subset["model_order"] = subset["model"].map(model_order)
                sort_cols = ["model_order", "oppu", "ft", "context", "style"]
                subset = subset.sort_values(by=sort_cols)
                subset["label"] = pd.Categorical(subset["label"], categories=subset["label"], ordered=True)

                sns.barplot(
                    ax=ax,
                    x="label", y=metric,
                    hue="model", data=subset,
                    dodge=False,
                    palette=MODEL_PALETTE
                )

                label_to_short = dict(zip(subset["label"], subset["short_label"]))
                ax.set_xticklabels(
                    [label_to_short.get(lbl.get_text(), lbl.get_text()) for lbl in ax.get_xticklabels()],
                    rotation=90
                )
                ax.set_ylim(0, 1.0)
                ax.set_title(f"{title}", pad=10, fontsize=24)
                if (i==1):
                    ax.set_xlabel("Model Configuration", fontsize=20)
                ax.set_ylabel(f"{title}", fontsize=20)

                # Capture handles and labels only once
                if not all_handles:
                    handles, labels = ax.get_legend_handles_labels()
                    all_handles, all_labels = handles, labels

                ax.get_legend().remove()

            # Deduplicate legend
            from collections import OrderedDict
            legend_dict = OrderedDict(zip(all_labels, all_handles))

            legend = fig.legend(
                legend_dict.values(), legend_dict.keys(),
                loc='upper right',
                bbox_to_anchor=(0.62, 0.7),
                title="Model",
                fontsize="small",
                title_fontsize="small",
                ncol=2,
            )
            legend.get_frame().set_alpha(0.8)
            legend.get_frame().set_facecolor("white")
            legend.get_frame().set_edgecolor("none")

            fig.tight_layout()
            save_figure(fig, f"accuracy_{response_type}_stacked")
            plt.close(fig)



def plot_accuracy_improvement_diff(optimal_df, random_df, metric="accuracy"):
    assert metric in ["accuracy", "class_0_accuracy"], "Metric must be 'accuracy' or 'class_0_accuracy'"
    apply_presentation_style()

    merged_df = pd.merge(
        optimal_df, random_df,
        on=["model", "style", "context", "ft", "oppu"],
        suffixes=("_opt", "_rand")
    )
    merged_df[f"{metric}_diff"] = merged_df[f"{metric}_opt"] - merged_df[f"{metric}_rand"]
    merged_df["label"] = merged_df["label_opt"]
    merged_df["short_label"] = merged_df["short_label_opt"]
    label_order = sorted(merged_df["label"].unique())
    hue_order = sorted(merged_df["model"].unique())

    with with_plot_style():
        fig, ax = plt.subplots(figsize=(14, 10))
        sns.barplot(
            x="label",
            y=f"{metric}_diff",
            hue="model",
            data=merged_df,
            dodge=False,
            palette=MODEL_PALETTE,
            order=label_order,
            hue_order=hue_order,
            ax=ax
        )

        title_map = {
            "accuracy": "Overall Accuracy Improvement (Optimal − Random)",
            "class_0_accuracy": "Class-0 (AI-generated text) Accuracy Improvement (Optimal − Random)"
        }
        ax.legend(title="Model")
        ax.set_title(title_map[metric], fontsize=24, pad=20,
                    color="white" if PRESENTATION_MODE else "black")
        ax.set_xlabel("")
        ax.set_ylabel("")

        ax.tick_params(axis='y', labelsize=16)
        label_to_short = dict(zip(merged_df["label"], merged_df["short_label"]))
        #ax.set_xticklabels([label_to_short[label.get_text()] for label in ax.get_xticklabels()], rotation=90)
        ax.set_xticklabels([])

        fig.tight_layout()
        save_figure(fig, f"{metric}_improvement_optimal_vs_random")
        plt.close(fig)


def plot_accuracy_improvement_diff_stacked(optimal_df, random_df):
    apply_presentation_style()

    # Merge optimal and random
    merged_df = pd.merge(
        optimal_df, random_df,
        on=["model", "style", "context", "ft", "oppu"],
        suffixes=("_opt", "_rand")
    )

    # Compute both diffs
    merged_df["accuracy_diff"] = merged_df["accuracy_opt"] - merged_df["accuracy_rand"]
    merged_df["class_0_accuracy_diff"] = merged_df["class_0_accuracy_opt"] - merged_df["class_0_accuracy_rand"]
    merged_df["label"] = merged_df["label_opt"]
    merged_df["short_label"] = merged_df["short_label_opt"]

    label_order = sorted(merged_df["label"].unique())
    hue_order = sorted(merged_df["model"].unique())
    label_to_short = dict(zip(merged_df["label"], merged_df["short_label"]))

    metric_titles = {
        "accuracy_diff": "Overall Accuracy Improvement (Optimal − Random)",
        "class_0_accuracy_diff": "Class-0 (AI-generated text) Accuracy Improvement (Optimal − Random)",
    }

    with with_plot_style():
        fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=True, sharey=True)

        all_handles = []
        all_labels = []

        for i, (metric, title) in enumerate(metric_titles.items()):
            ax = axes[i]

            sns.barplot(
                x="label",
                y=metric,
                hue="model",
                data=merged_df,
                dodge=False,
                palette=MODEL_PALETTE,
                order=label_order,
                hue_order=hue_order,
                ax=ax
            )

            ax.set_title(title, fontsize=24, pad=10,
                         color="white" if PRESENTATION_MODE else "black")
            ax.set_ylabel("")
            if i == 1:
                ax.set_xlabel("Model Configuration", fontsize=20)
            else:
                ax.set_xlabel("")

            ax.tick_params(axis='y', labelsize=16)
            ax.set_xticklabels([
                label_to_short.get(label.get_text(), label.get_text())
                for label in ax.get_xticklabels()
            ], rotation=90)

            if not all_handles:
                handles, labels = ax.get_legend_handles_labels()
                all_handles, all_labels = handles, labels

            ax.get_legend().remove()

        # Single legend
        from collections import OrderedDict
        legend_dict = OrderedDict(zip(all_labels, all_handles))
        legend = fig.legend(
            legend_dict.values(), legend_dict.keys(),
            loc='upper right',
            bbox_to_anchor=(0.5, 0.92),
            title="Model",
            fontsize="small",
            title_fontsize="small",
            ncol=2,
        )
        legend.get_frame().set_alpha(0.8)
        legend.get_frame().set_facecolor("white")
        legend.get_frame().set_edgecolor("none")

        fig.tight_layout()
        fig.subplots_adjust(top=0.92, right=0.88)  # Make room for legend
        save_figure(fig, "accuracy_improvement_optimal_vs_random_stacked")
        plt.close(fig)



def plot_scatter_accuracy(df):
    apply_presentation_style()

    for response_type in ["optimal", "random"]:
        scatter_df = df[df["response_type"] == response_type].dropna(subset=["class_0_accuracy"])

        with with_plot_style():
            fig, ax = plt.subplots(figsize=(12, 10))

            for model in scatter_df["model"].unique():
                model_data = scatter_df[scatter_df["model"] == model]
                for _, row in model_data.iterrows():
                    ax.scatter(
                        row["accuracy"],
                        row["class_0_accuracy"],
                        color=MODEL_PALETTE[model],
                        label=model if row["label"] == model_data.iloc[0]["label"] else "",
                        s=700,
                        alpha=0.8,
                        edgecolor="k",
                        marker=get_marker(row)
                    )

            ax.plot([0.5, 1.0], [0.5, 1.0], ls="--", color="gray")

            # Legends
            model_handles = [
                mlines.Line2D([], [], color=MODEL_PALETTE[model], marker='o', linestyle='None',
                            markersize=10, label=model)
                for model in sorted(MODEL_PALETTE)
            ]

            marker_labels = {
                'o': 'SE + CR + FT', #'Style 10, Context 3, FT 1',
                'd': 'SE + CR', #'Style 10, Context 3, FT 0',
                's': 'SE', #'Style 10, Context 0, FT 0',
                '^': 'baseline' #'Style 0, Context 0, FT 0',
            }
            marker_handles = [
                mlines.Line2D([], [], color="white" if PRESENTATION_MODE else "black", 
                            marker=marker, linestyle='None',
                            markersize=10, label=label)
                for marker, label in marker_labels.items()
            ]

        
            leg1 = ax.legend(handles=model_handles, title="Baseline Model", loc="lower right", 
                            frameon=True, framealpha=0.1, facecolor='white', edgecolor='black')
            leg2 = ax.legend(handles=marker_handles, title='Model configuration', loc='upper left',
                            frameon=True, framealpha=0.1, facecolor='white', edgecolor='black')
            ax.add_artist(leg1)

            # Labels
            ax.set_xlabel("Overall Accuracy", fontsize=20, color="white" if PRESENTATION_MODE else "black")
            ax.set_ylabel("Class-0 (AI-generated text) Accuracy", fontsize=20, color="white" if PRESENTATION_MODE else "black")
            ax.set_title("Overall Accuracy vs. Class-0 (AI-generated text) Accuracy", fontsize=24,
                        color="white" if PRESENTATION_MODE else "black")

            ax.set_xlim(0.49, 1.01)
            ax.set_ylim(0.49, 1.01)
            ax.tick_params(axis='both', which='major', labelsize=20)

            fig.tight_layout()
            save_figure(fig, f"scatter_accuracy_vs_class0_{response_type}")
            plt.close(fig)

In [124]:
df = load_confusion_data(FOLDER_PATH)

plot_all_accuracy_tiled(df)
plot_accuracy_by_response_type(df)
plot_scatter_accuracy(df)
optimal_df = df[df["response_type"] == "optimal"]
random_df = df[df["response_type"] == "random"]
plot_accuracy_improvement_diff(optimal_df, random_df, metric="accuracy")
plot_accuracy_improvement_diff(optimal_df, random_df, metric="class_0_accuracy")
plot_accuracy_improvement_diff_stacked(optimal_df, random_df)

/var/folders/nx/vkg436q12hd36l9v_4tzqh180000gn/T/ipykernel_31451/1665798199.py:93: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_xticklabels([label_to_short.get(lbl.get_text(), lbl.get_text()) for lbl in ax.get_xticklabels()], rotation=90)
/var/folders/nx/vkg436q12hd36l9v_4tzqh180000gn/T/ipykernel_31451/1665798199.py:93: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_xticklabels([label_to_short.get(lbl.get_text(), lbl.get_text()) for lbl in ax.get_xticklabels()], rotation=90)
/var/folders/nx/vkg436q12hd36l9v_4tzqh180000gn/T/ipykernel_31451/1665798199.py:93: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_xticklabels([label_to_short.get(lbl.get_text(), lbl.get_text()) for lbl in ax.get_xticklabels()], rotation=90)
/var/folders/nx/vkg436q12hd36l9v_4tzqh180000gn/T/ipykernel_31451/1665798199.py:93: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set

# EMPATH

In [127]:
RESULTS_FOLDER = "Figures/empath"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

# === Core plotting function ===
def plot_heatmap_significant_features(input_folder, response_type, value_column='adjusted_p_value'):
    apply_presentation_style()

    data, label_info, all_features = [], [], set()

    # === Traverse files ===
    for root, _, files in os.walk(input_folder):
        for file in files:
            if not file.endswith("empath_significant_features.csv"):
                continue

            # Match only relevant response type
            if response_type == "optimal" and "optimal_response" not in file and "optimal_response" not in root:
                continue
            if response_type == "random" and "random_response" not in file and "random_response" not in root:
                continue

            filepath = os.path.join(root, file)

            try:
                model, ft, context, style, oppu = parse_filename(file)
                label = make_label(model, ft, context, style, oppu)

                df = pd.read_csv(filepath)
                sig_df = df[df["adjusted_p_value"] < 0.05][["feature", value_column]]

                # Log-transformed p-value or raw difference
                if value_column == "adjusted_p_value":
                    row_dict = {
                        row["feature"]: -np.log10(max(row["adjusted_p_value"], 1e-10))
                        for _, row in sig_df.iterrows()
                    }
                else:
                    row_dict = {
                        row["feature"]: row["difference"]
                        for _, row in sig_df.iterrows()
                    }

                data.append(row_dict)
                label_info.append((label, model))
                all_features.update(sig_df["feature"].tolist())

            except Exception as e:
                print(f"⚠️ Error processing {filepath}: {e}")

    if not data:
        print(f"⚠️ No data found for response type '{response_type}'. Skipping heatmap.")
        return

    print(f"✅ Found {len(all_features)} unique features for '{response_type}'")

    # === Construct DataFrame ===
    heatmap_df = pd.DataFrame(data)
    labels, models = zip(*label_info)
    heatmap_df.index = labels
    heatmap_df.columns = [col.replace("_", " ") for col in heatmap_df.columns]

    # Sort by model + label for consistent color mapping
    sorting_index = sorted(range(len(labels)), key=lambda i: (models[i], labels[i]))
    heatmap_df = heatmap_df.iloc[sorting_index]
    sorted_labels = [labels[i] for i in sorting_index]
    sorted_models = [models[i] for i in sorting_index]
    sorted_colors = [MODEL_PALETTE.get(model, "gray") for model in sorted_models]

    # === Plot ===
    fig_width = min(25, 0.6 * heatmap_df.shape[1] + 5)
    fig_height = max(6, 0.3 * heatmap_df.shape[0])
    with with_plot_style():
        plt.figure(figsize=(fig_width, fig_height))

        cmap = "coolwarm" if value_column == "adjusted_p_value" else "vlag"
        colorbar_label = r"$-\log_{10}$(p-value)" if value_column == "adjusted_p_value" else "Difference"

        ax = sns.heatmap(
            heatmap_df,
            cmap='viridis',
            linewidths=0.3,
            linecolor='gray'
        )

        # === Format colorbar ===
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=14)
        cbar.set_label(colorbar_label, fontsize=16)

        # === Format axes ===
        ax.set_xticklabels(ax.get_xticklabels(), fontsize=16,  rotation=45, ha='right')
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)

        ax.set_title(f"Statistically different features, p-value",
                    fontsize=20, pad=15)
        ax.set_xlabel("")
        ax.set_ylabel("")

        # === Color y-tick labels by model ===
        for tick, color in zip(ax.get_yticklabels(), sorted_colors):
            tick.set_color(color)

        plt.tight_layout()

        # === Save output ===
        filename = f"significant_features_heatmap_{value_column}_{response_type}.png"
        output_path = os.path.join(RESULTS_FOLDER, filename)
        plt.savefig(output_path, dpi=600, transparent=PRESENTATION_MODE)
        plt.close()
    print(f"📊 Heatmap saved to {output_path}")

In [128]:
plot_heatmap_significant_features(FOLDER_PATH, 'random')
plot_heatmap_significant_features(FOLDER_PATH, 'optimal')

✅ Found 18 unique features for 'random'
📊 Heatmap saved to Figures/empath/significant_features_heatmap_adjusted_p_value_random.png
✅ Found 27 unique features for 'optimal'
📊 Heatmap saved to Figures/empath/significant_features_heatmap_adjusted_p_value_optimal.png


# COSINE SIMILARITY

In [89]:
import os
import json
import argparse
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from plotting_utils import MODEL_PALETTE, parse_filename, make_label  # import your external function here


def load_data(input_folder: Path):
    rows = []
    
    for dirpath, _, filenames in os.walk(input_folder):
        for fname in filenames:
            if fname.endswith("_with_cosine_scores.json"):
                file_path = Path(dirpath) / fname
                try:
                    # Use parse_filename to extract metadata from filename
                    model, ft, context, style, oppu = parse_filename(fname)
                    with open(file_path, "r", encoding="utf-8") as f:
                        data = json.load(f)
                        for entry in data:
                            sim = entry.get("original_vs_response_similarity")
                            if sim is None:
                                continue
                            label = make_label(model, ft, context, style, oppu, with_model=True)
                            rows.append({
                                "model": model,
                                "ft": ft,
                                "context": context,
                                "style": style,
                                "oppu": oppu,
                                "label": label,
                                "similarity": sim,
                            })
                except Exception as e:
                    print(f"⚠️ Failed to load {file_path.name}: {e}")
    print(f"✅ Done. Loaded {len(rows)} rows.")
    return pd.DataFrame(rows)

def plot_boxplot(df, output_path):
    apply_presentation_style()
    with with_plot_style():
        plt.figure(figsize=(18, 7))

        # Sort labels (x-axis)
        label_order = sorted(df["label"].unique())

        # Sort models (hue/legend)
        model_order = sorted(df["model"].unique())

        ax = sns.boxplot(
            data=df,
            x="label",
            y="similarity",
            hue="model",
            dodge=False,
            palette=MODEL_PALETTE,
            order=label_order,
            hue_order=model_order,  # <- Add this line to control hue order
            linewidth=1.0,
        )
        ax.set_xticklabels([])

        plt.title("Original vs Response Similarity by Model Configuration", fontsize=16)
        plt.ylabel("Cosine Similarity", fontsize=14)
        plt.xlabel("Configuration", fontsize=14)
        plt.xticks(rotation=90, ha="right")

        # Alphabetically sorted legend
        handles, labels = ax.get_legend_handles_labels()
        sorted_legend = sorted(zip(labels, handles), key=lambda x: x[0])
        labels, handles = zip(*sorted_legend)
        ax.legend(handles, labels, title="Model", bbox_to_anchor=(1.01, 1), loc='upper left')

        plt.tight_layout()
        output_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_path, dpi=600, bbox_inches="tight", transparent=PRESENTATION_MODE)
        plt.close()
        
    print(f"📊 Saved plot to: {output_path}")



def find_files(base_dir):
    # Scan all subfolders, find pairs of files
    files = {}
    for root, _, filenames in os.walk(base_dir):
        for f in filenames:
            if f.endswith("optimal_response.json"):
                prefix = f.replace("optimal_response.json", "")
                files.setdefault(prefix, {})["optimal"] = os.path.join(root, f)
            elif f.endswith("random_response_with_cosine_scores.json"):
                prefix = f.replace("random_response_with_cosine_scores.json", "")
                files.setdefault(prefix, {})["cosine"] = os.path.join(root, f)
    return files

def extract_similarity(optimal_path, cosine_path):
    with open(optimal_path, 'r') as f:
        optimal_data = json.load(f)
    with open(cosine_path, 'r') as f:
        cosine_data = json.load(f)
    
    model, ft, context, style, oppu = parse_filename(Path(cosine_path).name)
    label = make_label(model, ft, context, style, oppu, with_model=True)

    records = []
    for opt_entry in optimal_data:
        user = opt_entry.get("user")
        reply_to = opt_entry.get("reply_to")
        optimal_response = opt_entry.get("response")

        
        # Find matching entry in cosine data (same user and maybe 'reply_to'?)
        # Assuming same order for simplicity, but better to match by user & reply_to
        cosine_entry = next((e for e in cosine_data if e["user"] == user and e["reply_to"]==reply_to), None)
        if not cosine_entry:
            continue
        
        # Find similarity of optimal_response in valid_response_similarities
        similarity = None
        for sim_obj in cosine_entry.get("valid_response_similarities", []):
            if sim_obj["candidate"] == optimal_response:
                similarity = sim_obj["similarity"]
                break
        
        if similarity is not None:
            records.append({
                "user": user,
                "reply_to": reply_to,
                "model": model,
                "ft": ft,
                "context": context,
                "style": style,
                "oppu": oppu,
                "response": optimal_response,
                "similarity": similarity,
                "label": label  
            })
    return records

def load_all(base_dir):
    files = find_files(base_dir)
    all_records = []
    for prefix, file_dict in files.items():
        if "optimal" in file_dict and "cosine" in file_dict:
            recs = extract_similarity(file_dict["optimal"], file_dict["cosine"])
            all_records.extend(recs)
    return pd.DataFrame(all_records)


input_path = Path(FOLDER_PATH)
df = load_data(input_path)

if df.empty:
    print("❌ No similarity data found.")
else:
    print(f"✅ Loaded {len(df)} rows from {df['label'].nunique()} configurations.")
    plot_path = input_path / "Figures/similarity" / "similarity_boxplot_per_config.png"
    plot_boxplot(df, plot_path)

df = load_all(input_path)
if df.empty:
    print("❌ No similarity data found.")
else:
    print(f"✅ Loaded {len(df)} rows from {df['label'].nunique()} configurations.")
    plot_path = input_path / "Figures/similarity" / "optimal_similarity_boxplot_per_config.png"
    plot_boxplot(df, plot_path)


✅ Done. Loaded 140000 rows.
✅ Loaded 140000 rows from 28 configurations.
📊 Saved plot to: Figures/similarity/similarity_boxplot_per_config.png
✅ Loaded 139395 rows from 28 configurations.
📊 Saved plot to: Figures/similarity/optimal_similarity_boxplot_per_config.png
